# Dynamic Similarity for Compositional Image Retrieval

This is the **single final notebook** for the project. It loads the final trained system from `final_best_system/`, shows the official metrics and qualitative outputs, and contains the training/evaluation commands behind explicit boolean switches.

The final model keeps CLIP frozen and learns only a lightweight source-conditioned composition module. The winning inference formula is:

```text
q_model = learned_gate(source, query)
q_sum   = generic CLIP arithmetic(source, query)
q_final = normalize(q_model + 1.5 * (q_sum - source))
```

The notebook is designed to be run quickly by default. Heavy operations such as training and full official JSON evaluation are present but disabled unless the boolean flags are changed.

In [ ]:
# Runtime switches. Default values are safe for a quick notebook run.
RUN_INSTALL_DEPS = False
RUN_TRAINING = False
RUN_FULL_JSON_EVALUATION = False
RUN_QUALITATIVE_EXAMPLES = True
RUN_ORACLE_DIAGNOSTIC_SMOKE = False

DEVICE_REQUEST = "auto"
FINAL_BETA = 1.5

# Example used for the quick inference cell.
EXAMPLE_QUERY_ID = 5
EXAMPLE_SOURCE_INDEX = 3
EXAMPLE_TOP_K = 10

In [ ]:
if RUN_INSTALL_DEPS:
    import subprocess
    import sys
    subprocess.run([
        sys.executable, "-m", "pip", "install",
        "torch", "torchvision", "pandas", "matplotlib", "pillow", "tqdm"
    ], check=True)
else:
    print("Dependency installation skipped. Set RUN_INSTALL_DEPS=True only if needed.")

## Project Paths and Imports

The notebook searches upward from the current working directory until it finds `final_best_system/` and `cluster/`. This makes it runnable both from the repository root and from the `notebooks/` directory.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import Image as IPyImage, display


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "final_best_system").is_dir() and (candidate / "cluster").is_dir():
            return candidate
    raise RuntimeError("Could not find a project root containing final_best_system/ and cluster/.")


PROJECT_ROOT = find_project_root()
CLUSTER_ROOT = PROJECT_ROOT / "cluster"
FINAL_DIR = PROJECT_ROOT / "final_best_system"
FINAL_CODE = FINAL_DIR / "code"
FINAL_SCRIPTS = FINAL_CODE / "scripts"
FINAL_ORCH = FINAL_CODE / "orchestrator"

# The final-system helper modules use DL_PROJECT_ROOT to locate CelebA annotations and embeddings.
os.environ["DL_PROJECT_ROOT"] = str(CLUSTER_ROOT)
sys.path.insert(0, str(FINAL_SCRIPTS))
sys.path.insert(0, str(FINAL_ORCH))

from learned_gate_core import condition_embeddings, load_model_checkpoint, load_prompt_embedding_cache
from project_core import choose_device, load_torch, parse_query, read_attribute_table

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FINAL_DIR:", FINAL_DIR)

## Assignment Metric Definition

The assignment defines `Recall@K` as a **hit-rate**:

```text
Recall@K = 1 if at least one official-valid target appears in top-K, otherwise 0
Precision@K = number of official-valid targets in top-K / K
```

Therefore, a method can have good Recall@10 while still having low Precision@10. This means it often finds at least one valid target, but does not fill the entire top-10 with valid targets.

## Load the Final System

This cell loads:

- the final trained checkpoint from `final_best_system/weights/`;
- the signed prompt embedding cache used by the gate;
- the CLIP text directions used by the arithmetic correction;
- the frozen test image embedding gallery.

In [ ]:
DEVICE = choose_device(DEVICE_REQUEST)
print("Device:", DEVICE)

FINAL_CHECKPOINT_PATH = FINAL_DIR / "weights" / "best_val_official_like_at10.pt"
PROMPT_CACHE_PATH = FINAL_DIR / "embeddings" / "signed_attribute_prompt_embeddings_v2_photo_templates.pt"
TEXT_DIRECTIONS_PATH = FINAL_DIR / "embeddings" / "attribute_text_embeddings.pt"
TEST_EMBEDDINGS_PATH = CLUSTER_ROOT / "data" / "celeba" / "embeddings" / "openai_clip_vit_b32" / "test_image_embeddings.pt"
EVAL_JSON_PATH = PROJECT_ROOT / "celeba_evaluation.json"
if not EVAL_JSON_PATH.exists():
    EVAL_JSON_PATH = CLUSTER_ROOT / "data" / "celeba_evaluation.json"

required = [FINAL_CHECKPOINT_PATH, PROMPT_CACHE_PATH, TEXT_DIRECTIONS_PATH, TEST_EMBEDDINGS_PATH, EVAL_JSON_PATH]
for p in required:
    assert p.exists(), f"Missing required file: {p}"

final_model, final_checkpoint = load_model_checkpoint(FINAL_CHECKPOINT_PATH, DEVICE)
final_model.eval()
final_config = final_checkpoint["config"]

prompt_cache = load_prompt_embedding_cache(PROMPT_CACHE_PATH)
text_bank = load_torch(TEXT_DIRECTIONS_PATH)
text_directions = F.normalize(text_bank["directions"].float().to(DEVICE), dim=-1)

gallery_cache = load_torch(TEST_EMBEDDINGS_PATH)
gallery = F.normalize(gallery_cache["embeddings"].float().to(DEVICE), dim=-1)
gallery_filenames = list(gallery_cache["filenames"])

attributes, _, _ = read_attribute_table()
attribute_to_index = {name: i for i, name in enumerate(attributes)}
evaluation_json = json.loads(EVAL_JSON_PATH.read_text())

print("Checkpoint:", FINAL_CHECKPOINT_PATH)
print("Prompt cache:", PROMPT_CACHE_PATH.name)
print("Gallery:", gallery.shape)
print("Official JSON query entries:", len(evaluation_json))

## Final Inference Function

For each source image and textual modification query, the system computes:

```text
q_model = learned_gate(source, query)
q_sum   = normalize(source + sum signed CLIP text directions)
q_final = normalize(q_model + beta * (q_sum - source))
```

Retrieval then ranks every gallery image by cosine similarity with `q_final`.

In [ ]:
def condition_tensors_for_query(conditions, batch_size, device):
    max_len = max(1, len(conditions))
    attrs = torch.full((batch_size, max_len), -1, dtype=torch.long, device=device)
    signs = torch.zeros((batch_size, max_len), dtype=torch.int8, device=device)
    for pos, (sign, attr) in enumerate(conditions):
        attrs[:, pos] = attribute_to_index[attr]
        signs[:, pos] = int(sign)
    return attrs, signs


def learned_gate_query(source_embeddings, conditions):
    source_embeddings = F.normalize(source_embeddings.float().to(DEVICE), dim=-1)
    attrs, signs = condition_tensors_for_query(conditions, len(source_embeddings), DEVICE)
    cond, mask = condition_embeddings(
        prompt_cache,
        attrs,
        signs,
        DEVICE,
        str(final_config.get("condition_mode", "signed_direction")),
    )
    with torch.inference_mode():
        q_model, alpha, _ = final_model(source_embeddings, cond, mask)
    return F.normalize(q_model, dim=-1), alpha


def generic_sum_query(source_embeddings, conditions):
    source_embeddings = F.normalize(source_embeddings.float().to(DEVICE), dim=-1)
    edit = torch.zeros_like(source_embeddings)
    for sign, attr in conditions:
        edit = edit + int(sign) * text_directions[attribute_to_index[attr]].unsqueeze(0)
    edit = F.normalize(edit, dim=-1)
    return F.normalize(source_embeddings + edit, dim=-1)


def final_query(source_embeddings, query_text, beta=FINAL_BETA):
    conditions = parse_query(query_text)
    q_model, alpha = learned_gate_query(source_embeddings, conditions)
    q_sum = generic_sum_query(source_embeddings, conditions)
    q_final = F.normalize(q_model + float(beta) * (q_sum - F.normalize(source_embeddings.float().to(DEVICE), dim=-1)), dim=-1)
    return q_final, {"conditions": conditions, "alpha": alpha.detach().cpu(), "q_model": q_model.detach().cpu(), "q_sum": q_sum.detach().cpu()}


def retrieve_topk_for_source(query_id, source_index, top_k=10, beta=FINAL_BETA):
    query_text = evaluation_json[query_id]["query"]
    source = gallery[[source_index]]
    q_final, details = final_query(source, query_text, beta=beta)
    scores = q_final @ gallery.T
    scores[0, source_index] = -torch.inf
    top = scores.topk(top_k, dim=1).indices[0].detach().cpu().tolist()
    valid_targets = set(map(int, evaluation_json[query_id]["ground_truth"].get(str(source_index), [])))
    rows = []
    for rank, idx in enumerate(top, start=1):
        rows.append({
            "rank": rank,
            "test_index": idx,
            "filename": gallery_filenames[idx],
            "official_json_valid": idx in valid_targets,
        })
    return pd.DataFrame(rows), details

## Quick Inference Smoke Test

This cell recomputes the final vector from the final checkpoint and retrieves top-10 images for one official JSON source/query case. It does **not** use the precomputed `retrievals.jsonl`.

In [ ]:
query_text = evaluation_json[EXAMPLE_QUERY_ID]["query"]
print(f"Example query_id={EXAMPLE_QUERY_ID}: {query_text}")
print(f"Source index: {EXAMPLE_SOURCE_INDEX}, filename: {gallery_filenames[EXAMPLE_SOURCE_INDEX]}")

example_topk, example_details = retrieve_topk_for_source(EXAMPLE_QUERY_ID, EXAMPLE_SOURCE_INDEX, EXAMPLE_TOP_K)
display(example_topk)
print("Gate alpha values:", example_details["alpha"].numpy().round(3).tolist())

## Official Results Saved in the Repository

The full official evaluation over all 33,052 source-query pairs has already been run on the cluster and copied into `final_best_system/results/`. The table below compares:

1. assignment baseline: direct CLIP arithmetic;
2. strongest zero-shot CLIP baseline: contrastive sequential arithmetic;
3. final learned system: learned gate + generic arithmetic correction.

In [ ]:
macro_metrics = pd.read_csv(FINAL_DIR / "results" / "clean_report" / "overall_metrics_macro.csv")
micro_metrics = pd.read_csv(FINAL_DIR / "results" / "clean_report" / "overall_metrics_micro.csv")
all_metrics = pd.read_csv(FINAL_DIR / "results" / "clean_report" / "overall_metrics_three_systems.csv")
per_query = pd.read_csv(FINAL_DIR / "results" / "clean_report" / "per_query_recall10_three_systems.csv")

print("Macro metrics: each query counts equally.")
display(macro_metrics)

print("Micro metrics: every source-query case counts equally.")
display(micro_metrics)

for image_name in [
    "overall_metrics_k10_focus.png",
    "overall_metrics_three_systems.png",
    "per_query_recall10_three_systems.png",
    "per_query_recall10_improvement.png",
]:
    image_path = FINAL_DIR / "results" / "clean_report" / image_name
    if image_path.exists():
        display(IPyImage(filename=str(image_path)))

## Qualitative Examples

The qualitative viewer shows:

- blue: source/input image;
- light green: official-valid JSON target;
- yellow: satisfies the requested query attributes, but is not JSON-valid;
- red: fails at least one requested query attribute.

This helps diagnose whether a retrieved image fails because it violates the requested edit or because it does not satisfy the stricter official Hamming/source-preservation rule.

In [ ]:
if RUN_QUALITATIVE_EXAMPLES:
    qualitative_examples = [
        (5, 3),      # +Blond_Hair
        (12, 37),    # -Smiling, +Eyeglasses, +Wearing_Hat
        (13, 3977),  # +Wearing_Lipstick, -Heavy_Makeup, +Smiling
    ]
    for query_id, source_index in qualitative_examples:
        subprocess.run(
            [
                sys.executable,
                str(FINAL_DIR / "code" / "show_json_retrieval_example.py"),
                "--query-id", str(query_id),
                "--source-index", str(source_index),
                "--top-k", "10",
                "--no-open",
            ],
            check=True,
            cwd=PROJECT_ROOT,
        )
        path = FINAL_DIR / "results" / "qualitative_examples" / f"query_{query_id:02d}_source_{source_index}_top10.png"
        display(IPyImage(filename=str(path)))
else:
    print("Qualitative examples skipped.")

## Training Code, Disabled by Default

The final model was trained on the cluster. The training pipeline is included here for reproducibility but is disabled by default because it can take hours.

The v6 training strategy mixes:

```text
60% official-like multi-positive pairs from train split, Hamming <= 2
30% same-identity source-preservation pairs
10% weak/global attribute oversampling
```

The official test JSON is **not** used to construct training pairs.

In [ ]:
# Toggle RUN_TRAINING=True at the top of the notebook to run this.
# On a laptop, keep profile="short" for a smoke test. On the cluster, use profile="long".

TRAINING_PROFILE = "short"  # "short" or "long"
TRAINING_CONFIG = CLUSTER_ROOT / "configs" / "gate_v6_official_mix_3h_configs.json"
TRAIN_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if RUN_TRAINING:
    commands = [
        [
            sys.executable,
            str(CLUSTER_ROOT / "experimental" / "build_official_like_positive_sets_v6.py"),
            "--split", "train",
            "--preset", "official",
            "--device", TRAIN_DEVICE,
            "--force",
        ],
        [
            sys.executable,
            str(CLUSTER_ROOT / "experimental" / "build_official_like_positive_sets_v6.py"),
            "--split", "train",
            "--preset", "weak",
            "--device", TRAIN_DEVICE,
            "--force",
        ],
        [
            sys.executable,
            str(CLUSTER_ROOT / "experimental" / "run_official_mix_hpsearch_v6.py"),
            "--profile", TRAINING_PROFILE,
            "--configs", str(TRAINING_CONFIG),
            "--device", TRAIN_DEVICE,
            "--limit", "1" if TRAINING_PROFILE == "short" else "0",
            "--force",
        ],
    ]
    for command in commands:
        print("Running:", " ".join(map(str, command)))
        subprocess.run(command, check=True, cwd=CLUSTER_ROOT)
else:
    print("Training skipped. Set RUN_TRAINING=True to execute the reproducibility training pipeline.")

## Full Official Evaluation Code, Disabled by Default

The final official evaluation is already saved. This cell shows how it can be recomputed with the final checkpoint and beta value.

In [ ]:
if RUN_FULL_JSON_EVALUATION:
    eval_device = "cuda" if torch.cuda.is_available() else "cpu"
    output_root = FINAL_DIR / "results" / "notebook_recomputed_beta_eval"
    subprocess.run(
        [
            sys.executable,
            str(FINAL_CODE / "orchestrator" / "evaluate_beta_sweep_blends.py"),
            "--checkpoint", str(FINAL_CHECKPOINT_PATH),
            "--output-root", str(output_root),
            "--betas", str(FINAL_BETA),
            "--device", eval_device,
            "--force",
        ],
        check=True,
        cwd=FINAL_CODE,
    )
    print("Saved recomputed evaluation to", output_root)
else:
    print("Full JSON evaluation skipped. Set RUN_FULL_JSON_EVALUATION=True to recompute it.")

## Oracle Top-Pool Diagnostic, Disabled by Default

This diagnostic retrieves a broad top-N candidate pool with `q_final` and then filters candidates using ground-truth CelebA attributes/Hamming constraints. This is **not** a fair inference method, because those labels would not be available in a general deployment. It is an upper-bound analysis showing whether `q_final` reaches a useful candidate region.

In [ ]:
if RUN_ORACLE_DIAGNOSTIC_SMOKE:
    subprocess.run(
        [
            sys.executable,
            str(FINAL_DIR / "code" / "test_oracle_top500_rerank.py"),
            "--query-ids", "13",
            "--source-index", "3977",
            "--top-pool", "500",
            "--top-k", "10",
            "--device", "cpu",
            "--output-dir", str(FINAL_DIR / "results" / "oracle_top500_hamming_filter" / "notebook_smoke_q13_source3977"),
        ],
        check=True,
        cwd=PROJECT_ROOT,
    )
    oracle_summary = pd.read_csv(FINAL_DIR / "results" / "oracle_top500_hamming_filter" / "notebook_smoke_q13_source3977" / "summary.csv")
    display(oracle_summary)
else:
    print("Oracle diagnostic skipped. Set RUN_ORACLE_DIAGNOSTIC_SMOKE=True for a small smoke test.")

## Findings

The final system improves substantially over both the assignment baseline and the strongest zero-shot CLIP arithmetic baseline.

```text
Assignment baseline Macro Recall@10:      0.108
Strong CLIP baseline Macro Recall@10:    0.187
Final system Macro Recall@10:            0.391
```

The remaining weakness is Precision@10: the model often reaches a useful region and retrieves at least one valid target, but it does not always fill all ten slots with official-valid targets. Qualitative and oracle-pool diagnostics suggest a future improvement: retrieve a broad candidate pool with `q_final`, then apply a fair learned reranker that estimates query satisfaction and source preservation without using test labels directly.